Верификация диктора

Сравниваем два WAV и возвращаем SAME / DIFFERENT с confidence. Используем ECAPA после адаптации на Vibravox; порог выбираем по validation с учётом цены ошибок. 

In [ ]:
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from threading import Lock
from uuid import uuid4
from hyperpyyaml import load_hyperpyyaml
import os
import stat
import gradio as gr
import numpy as np
import soundfile as sf
import torch
import torchaudio
import speechbrain
import hashlib
import hmac
import math
import tempfile
import logging


from speechbrain.inference.speaker import SpeakerRecognition
from speechbrain.utils.fetching import LocalStrategy


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Python:", sys.version.split()[0])
print("Система:", platform.platform())
print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)
print("SpeechBrain:", speechbrain.__version__)
print("Gradio:", gr.__version__)
print("Устройство:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Используем готовый checkpoint шага 500 и соответствующий калибратор. Для автоматического выбора порога нужны сохранённые validation-эмбеддинги и пары; test при подборе порога не используется.

Перед загрузкой проверяем целостность комплекта и согласованность модели, калибратора и validation-данных. При несовпадении запуск останавливается. Ожидаемые хеши фиксированы в ноутбуке; защита самого ноутбука и запрет записи в каталог модели будут обеспечены инфраструктурой

In [ ]:
project_dir = Path(
    r"C:\Users\niki\Desktop\speaker_verification"
).resolve(strict=True)

base_model_dir = project_dir / "pretrained_models/ecapa"
checkpoint_path = (
    project_dir
    / "checkpoints/vibravox_fc_classification_1200/best.pt"
)
calibration_path = (
    checkpoint_path.parent / "calibration_vibravox_candidate.json"
)

experiment_dir = project_dir / "experiments/pair_classifiers"
embedding_config_path = experiment_dir / "embedding_config.json"
validation_embeddings_path = experiment_dir / "validation_embeddings.npz"
validation_pairs_path = experiment_dir / "validation_pair_indices.npz"

max_file_bytes = 5 * 1024 * 1024
min_audio_seconds = 1.0
max_audio_seconds = 5.0

inference_lock = Lock()

# Контрольная точка текущего комплекта.
# Обновляется вручную только при выпуске проверенной новой версии.
artifact_manifest = {
    "pretrained_models/ecapa/hyperparams.yaml": (
        1919,
        "6f78854fa04ba59e761437b76a2575d3aba5e5016de3e9b69f0c9a5077fb1a41",
    ),
    "pretrained_models/ecapa/embedding_model.ckpt": (
        83316686,
        "0575cb64845e6b9a10db9bcb74d5ac32b326b8dc90352671d345e2ee3d0126a2",
    ),
    "pretrained_models/ecapa/mean_var_norm_emb.ckpt": (
        1921,
        "cd70225b05b37be64fc5a95e24395d804231d43f74b2e1e5a513db7b69b34c33",
    ),
    "pretrained_models/ecapa/classifier.ckpt": (
        5534328,
        "fd9e3634fe68bd0a427c95e354c0c677374f62b3f434e45b78599950d860d535",
    ),
    "pretrained_models/ecapa/label_encoder.ckpt": (
        128619,
        "e13c3a167bb4112685670ee896d20e2b565af16b3a4ceeaa8689fa4d22adb8b9",
    ),
    "checkpoints/vibravox_fc_classification_1200/best.pt": (
        93067431,
        "1db78748d622b220fde76b8cf163a966517451ae9d1cc47fae8504bfa1bcc561",
    ),
    "checkpoints/vibravox_fc_classification_1200/calibration_vibravox_candidate.json": (
        2212,
        "e3f452fbb1eaa8baf2610630885b4a458969251482fa148a959e12ad395c1df4",
    ),
    "experiments/pair_classifiers/embedding_config.json": (
        394,
        "d781f8db4de8722d2b857cc18cfc323ce5192a11f38a7a6ccec93de197548ed0",
    ),
    "experiments/pair_classifiers/validation_embeddings.npz": (
        75796,
        "5578ff3a262bd6cdfe956d0ae9b8ca3f321df2e6df326de9b8068ce1d5914525",
    ),
    "experiments/pair_classifiers/validation_pair_indices.npz": (
        3638,
        "1c2a890ccfe39c998ab75b7b4552b7f92156e012c89708fdb347fe35c87a3063",
    ),
}


def verify_artifacts():
    for relative_path, (expected_size, expected_hash) in artifact_manifest.items():
        path = (project_dir / relative_path).resolve(strict=True)

        if not path.is_relative_to(project_dir):
            raise RuntimeError(
                f"Артефакт находится вне каталога проекта: {relative_path}"
            )

        if not path.is_file():
            raise RuntimeError(f"Ожидался обычный файл: {relative_path}")

        if path.stat().st_size != expected_size:
            raise RuntimeError(
                f"Изменился размер артефакта: {relative_path}"
            )

        digest = hashlib.sha256()

        with path.open("rb") as file:
            for chunk in iter(lambda: file.read(1024 * 1024), b""):
                digest.update(chunk)

        if not hmac.compare_digest(digest.hexdigest(), expected_hash):
            raise RuntimeError(
                f"SHA-256 не совпадает: {relative_path}. "
                "Загрузка остановлена. Не обновляй хеш автоматически."
            )

    # В этом комплекте дополнительный Python-код не предусмотрен.
    # В частности, не разрешаем незаметно добавить custom.py.
    expected_model_files = {
        Path(relative_path).name
        for relative_path in artifact_manifest
        if relative_path.startswith("pretrained_models/ecapa/")
    }

    actual_model_files = {
        path.name for path in base_model_dir.iterdir()
    }

    unexpected = actual_model_files - expected_model_files

    if unexpected:
        raise RuntimeError(
            "В каталоге исходной модели обнаружены лишние объекты: "
            + ", ".join(sorted(unexpected))
        )


def read_json_object(path):
    def reject_constant(value):
        raise ValueError(f"Недопустимая JSON-константа: {value}")

    def reject_duplicate_keys(pairs):
        result = {}

        for key, value in pairs:
            if key in result:
                raise ValueError(f"Повторный JSON-ключ: {key}")
            result[key] = value

        return result

    with path.open(encoding="utf-8") as file:
        result = json.load(
            file,
            parse_constant=reject_constant,
            object_pairs_hook=reject_duplicate_keys,
        )

    if not isinstance(result, dict):
        raise ValueError(f"Ожидался JSON-объект: {path.name}")

    return result


verify_artifacts()

calibration_config = read_json_object(calibration_path)
embedding_config = read_json_object(embedding_config_path)

for name, config in (
    ("Калибратор", calibration_config),
    ("Описание эмбеддингов", embedding_config),
):
    if config.get("step") != 500:
        raise ValueError(f"{name}: ожидался шаг 500")

    recorded_checkpoint = Path(config["checkpoint"]).resolve(strict=True)

    if recorded_checkpoint != checkpoint_path.resolve(strict=True):
        raise ValueError(f"{name}: указан другой checkpoint")

for key in ("a", "b", "decision_threshold", "assumed_same_prior"):
    value = calibration_config[key]

    if (
        isinstance(value, bool)
        or not isinstance(value, (int, float))
        or not math.isfinite(value)
    ):
        raise ValueError(f"Некорректный параметр калибратора: {key}")

if calibration_config["a"] <= 0:
    raise ValueError("Ожидался возрастающий калибратор")

if not -1 <= calibration_config["decision_threshold"] <= 1:
    raise ValueError("Порог cosine должен находиться в [-1, 1]")

if calibration_config["assumed_same_prior"] != 0.5:
    raise ValueError("Ожидалась калибровка с равными долями классов")

if calibration_config.get("equal_scenario_weights") is not True:
    raise ValueError("Ожидались равные веса акустических сценариев")

expected_embedding_settings = {
    "sample_rate_before_model": 16000,
    "max_seconds": 5,
    "crop": "first",
    "embedding_dimension": 192,
    "l2_normalized": True,
    "denoising": False,
    "pause_removal": False,
}

for key, expected in expected_embedding_settings.items():
    actual = embedding_config.get(key)

    if type(actual) is not type(expected) or actual != expected:
        raise ValueError(
            f"Несовместимая настройка эмбеддингов: {key}"
        )

print(f"Проверено артефактов: {len(artifact_manifest)}")
print("Размеры и SHA-256 совпадают.")
print("Пути находятся внутри проекта.")
print("Лишних объектов в каталоге исходной модели нет.")
print("Метаданные модели, калибратора и эмбеддингов согласованы.")
print("Загрузка весов ещё не выполнялась.")

Загружаем проверенную архитектуру и дообученные веса. weights_only=True ограничивает десериализацию checkpoint, а strict=True проверяет соответствие весов архитектуре. Модель переводим в режим оценки и запрещаем обучение.

In [ ]:
verify_artifacts()

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

if not isinstance(checkpoint, dict):
    raise ValueError("Ожидался checkpoint в формате словаря")

if checkpoint.get("step") != 500:
    raise ValueError("Ожидался checkpoint шага 500")

encoder_state = checkpoint.get("encoder_state")

if not isinstance(encoder_state, dict) or not encoder_state:
    raise ValueError("В checkpoint отсутствуют веса encoder")

for name, value in encoder_state.items():
    if not isinstance(name, str) or not isinstance(value, torch.Tensor):
        raise ValueError("Некорректная структура весов encoder")

    if not torch.isfinite(value).all().item():
        raise ValueError(f"Неконечные значения в весах: {name}")

checkpoint_threshold = float(
    checkpoint["validation"]["threshold"]
)

if not math.isclose(
    checkpoint_threshold,
    calibration_config["decision_threshold"],
    rel_tol=0,
    abs_tol=1e-9,
):
    raise ValueError("Пороги checkpoint и калибратора не совпадают")

# YAML создаёт Python-объекты: используем только проверенный файл.
# Pretrainer не запускаем, скачивание и загрузку исходных весов не вызываем.
with (base_model_dir / "hyperparams.yaml").open(
    encoding="utf-8"
) as file:
    model_config = load_hyperpyyaml(
        file,
        overrides={
            "pretrained_path": base_model_dir.as_posix(),
        },
    )

inference_modules = torch.nn.ModuleDict({
    "compute_features": model_config["compute_features"],
    "mean_var_norm": model_config["mean_var_norm"],
    "embedding_model": model_config["embedding_model"],
})

encoder = inference_modules["embedding_model"]
encoder.load_state_dict(encoder_state, strict=True)

inference_modules.to(device)
inference_modules.eval()
inference_modules.requires_grad_(False)

calibration_a = float(calibration_config["a"])
calibration_b = float(calibration_config["b"])
reference_threshold = float(
    calibration_config["decision_threshold"]
)

del encoder_state, checkpoint, model_config

print("ECAPA загружена: checkpoint шага 500")
print("Устройство:", next(encoder.parameters()).device)
print("Веса соответствуют архитектуре.")
print(
    "Обучаемых параметров:",
    sum(
        parameter.numel()
        for parameter in inference_modules.parameters()
        if parameter.requires_grad
    ),
)
print(f"Исходный порог: {reference_threshold:.9f}")
print("Обучение и скачивание не выполнялись.")

Открываем WAV один раз, проверяем тип и размер файла, затем ограничиваем декодирование его длительностью и числом каналов. Проверка изменения файла во время чтения — дополнительная мера; полноценную защиту каталога загрузок обеспечит инфраструктура.

In [ ]:
def load_audio(path):
    if path is None:
        raise ValueError("Нужно передать WAV-файл")

    path = Path(path)

    try:
        with path.open("rb") as source:
            before = os.fstat(source.fileno())

            if not stat.S_ISREG(before.st_mode):
                raise ValueError("Ожидался обычный файл")

            if not 0 < before.st_size <= max_file_bytes:
                raise ValueError(
                    "WAV должен быть непустым и не превышать 5 МБ"
                )

            with sf.SoundFile(source, mode="r") as audio:
                sample_rate = audio.samplerate
                channels = audio.channels
                frames = audio.frames

                if audio.format not in {"WAV", "WAVEX", "RF64"}:
                    raise ValueError("Поддерживаются только WAV-файлы")

                if channels not in (1, 2):
                    raise ValueError("Поддерживаются mono и stereo")

                if not 8000 <= sample_rate <= 48000:
                    raise ValueError(
                        "Частота должна быть 8000–48000 Гц"
                    )

                if not (
                    min_audio_seconds * sample_rate
                    <= frames
                    <= max_audio_seconds * sample_rate
                ):
                    raise ValueError(
                        "Для этого прототипа нужна запись от 1 до 5 секунд"
                    )

                signal = audio.read(
                    frames=frames,
                    dtype="float32",
                    always_2d=True,
                )

            after = os.fstat(source.fileno())

            if (
                before.st_size != after.st_size
                or before.st_mtime_ns != after.st_mtime_ns
            ):
                raise ValueError("Файл изменился во время чтения")

    except sf.LibsndfileError as error:
        raise ValueError(
            "Не удалось декодировать WAV: файл повреждён "
            "или его кодирование не поддерживается"
        ) from error
    except OSError as error:
        raise ValueError(
            "Файл недоступен или произошла ошибка чтения"
        ) from error

    if signal.shape != (frames, channels):
        raise ValueError(
            "Число прочитанных отсчётов не совпало с заголовком"
        )

    if not np.isfinite(signal).all():
        raise ValueError("Аудиосигнал содержит NaN или Inf")

    mono = signal.mean(axis=1)

    if np.max(np.abs(mono)) < 1e-6:
        raise ValueError(
            "После преобразования в mono сигнал слишком слабый"
        )

    waveform = torch.from_numpy(mono.copy())

    if sample_rate != 8000:
        waveform = torchaudio.functional.resample(
            waveform,
            orig_freq=sample_rate,
            new_freq=8000,
        )

    waveform = torchaudio.functional.resample(
        waveform,
        orig_freq=8000,
        new_freq=16000,
    )

    if not torch.isfinite(waveform).all().item():
        raise ValueError("Некорректный сигнал после ресемплинга")

    return waveform.unsqueeze(0), frames / sample_rate


print("load_audio готова: ограниченное чтение через один открытый файл.")

Преобразуем подготовленную запись в эмбеддинг размерности 192 и нормализуем его длину. Используем режим инференса без вычисления градиентов; некорректный выход модели вызывает ошибку, а не результат сравнения.

In [ ]:
@torch.inference_mode()
def extract_embedding(waveform):
    if not isinstance(waveform, torch.Tensor):
        raise ValueError("Ожидался аудиосигнал в формате Tensor")

    if (
        waveform.ndim != 2
        or waveform.shape[0] != 1
        or waveform.dtype != torch.float32
    ):
        raise ValueError("Ожидался float32 Tensor формы (1, samples)")

    if not (
        min_audio_seconds * 16000
        <= waveform.shape[1]
        <= max_audio_seconds * 16000
    ):
        raise ValueError("Некорректная длина подготовленного сигнала")

    if not torch.isfinite(waveform).all().item():
        raise ValueError("Подготовленный сигнал содержит NaN или Inf")

    waveform = waveform.to(device)
    lengths = torch.ones(1, device=device)

    features = inference_modules["compute_features"](waveform)
    features = inference_modules["mean_var_norm"](features, lengths)

    vector = encoder(features, lengths).reshape(-1)

    if vector.numel() != 192:
        raise RuntimeError("Модель вернула неожиданную размерность")

    norm = vector.norm()

    if (
        not torch.isfinite(vector).all().item()
        or not torch.isfinite(norm).item()
        or norm.item() <= 1e-12
    ):
        raise RuntimeError("Модель вернула некорректный эмбеддинг")

    return vector / norm


print("extract_embedding готова. Размерность выхода: 192.")

Проверяем конвейер на сохранённой validation-записи: загрузку WAV, размерность и норму эмбеддинга, повторяемость инференса. Эта проверка подтверждает работоспособность конвейера, а не качество верификации.

In [ ]:
verify_artifacts()

with np.load(
    validation_embeddings_path,
    allow_pickle=False,
) as data:
    validation_paths = data["paths"].copy()

if validation_paths.ndim != 1 or len(validation_paths) == 0:
    raise ValueError("Некорректный список validation-записей")

if validation_paths.dtype.kind != "U":
    raise ValueError("Ожидались строковые пути validation-записей")

validation_audio_dir = (
    project_dir / "data/vibravox/validation"
).resolve(strict=True)

sample_path = Path(str(validation_paths[0])).resolve(strict=True)

if not sample_path.is_relative_to(validation_audio_dir):
    raise ValueError("Запись находится вне каталога validation")

waveform, duration = load_audio(sample_path)

with inference_lock:
    first_embedding = extract_embedding(waveform)
    second_embedding = extract_embedding(waveform)

if first_embedding.shape != (192,):
    raise RuntimeError("Неожиданная форма эмбеддинга")

embedding_norm = first_embedding.norm().item()
repeat_difference = (
    first_embedding - second_embedding
).abs().max().item()

if not math.isclose(
    embedding_norm,
    1.0,
    rel_tol=0,
    abs_tol=1e-5,
):
    raise RuntimeError("Эмбеддинг не нормализован")

if not torch.allclose(
    first_embedding,
    second_embedding,
    rtol=1e-4,
    atol=1e-5,
):
    raise RuntimeError("Повторный инференс дал существенно другой результат")

print("Файл:", sample_path.name)
print(f"Длительность: {duration:.2f} с")
print("Аудио:", tuple(waveform.shape), waveform.dtype)
print("Эмбеддинг:", tuple(first_embedding.shape))
print("Устройство:", first_embedding.device)
print(f"Норма: {embedding_norm:.6f}")
print(f"Максимальное отличие повторов: {repeat_difference:.8f}")
print("Проверка чтения и инференса пройдена.")

del first_embedding, second_embedding, waveform

Для выбора порога используем сохранённые validation-эмбеддинги модели шага 500. Проверяем размеры, метки классов и акустические сценарии. Дополнительно сравниваем эмбеддинг уже проверенной записи с кешем — это проверка согласованности текущего конвейера с сохранёнными данными.

In [ ]:
verify_artifacts()

with np.load(
    validation_embeddings_path,
    allow_pickle=False,
) as data:
    validation_embeddings = data["embeddings"].copy()
    validation_paths = data["paths"].copy()
    validation_speakers = data["speaker_ids"].copy()
    validation_conditions = data["conditions"].copy()

with np.load(
    validation_pairs_path,
    allow_pickle=False,
) as data:
    validation_pair_indices = {
        key: data[key].copy()
        for key in data.files
    }

record_count = len(validation_paths)

if validation_embeddings.shape != (record_count, 192):
    raise ValueError("Некорректная форма validation-эмбеддингов")

for name, values in (
    ("paths", validation_paths),
    ("speaker_ids", validation_speakers),
    ("conditions", validation_conditions),
):
    if values.shape != (record_count,) or values.dtype.kind != "U":
        raise ValueError(f"Некорректное поле: {name}")

if len(set(validation_paths.tolist())) != record_count:
    raise ValueError("В validation повторяются пути записей")

if not np.isfinite(validation_embeddings).all():
    raise ValueError("Эмбеддинги содержат NaN или Inf")

if not np.allclose(
    np.linalg.norm(validation_embeddings, axis=1),
    1.0,
    rtol=0,
    atol=1e-5,
):
    raise ValueError("Validation-эмбеддинги не нормализованы")

if set(validation_conditions.tolist()) != {
    "speech_clean",
    "speech_noisy",
}:
    raise ValueError("Неожиданные акустические условия")

# Проверяем соответствие текущего инференса сохранённому кешу.
sample_path = Path(str(validation_paths[0])).resolve(strict=True)

if not sample_path.is_relative_to(validation_audio_dir):
    raise ValueError("Запись находится вне каталога validation")

sample_audio, _ = load_audio(sample_path)

with inference_lock:
    current_embedding = (
        extract_embedding(sample_audio).cpu().numpy()
    )

cache_difference = float(np.max(np.abs(
    current_embedding - validation_embeddings[0]
)))

if not np.allclose(
    current_embedding,
    validation_embeddings[0],
    rtol=1e-4,
    atol=1e-5,
):
    raise RuntimeError(
        "Текущий инференс не совпал с validation-кешем. "
        "Проверь модель и обработку аудио."
    )

scenario_conditions = {
    "clean-clean": ("speech_clean", "speech_clean"),
    "clean-noisy": ("speech_clean", "speech_noisy"),
    "noisy-noisy": ("speech_noisy", "speech_noisy"),
}

validation_scores = {}

for scenario, (first_condition, second_condition) in scenario_conditions.items():
    first = validation_pair_indices[f"{scenario}_first"]
    second = validation_pair_indices[f"{scenario}_second"]
    labels = validation_pair_indices[f"{scenario}_labels"]

    if (
        first.ndim != 1
        or first.size == 0
        or first.shape != second.shape
        or first.shape != labels.shape
        or first.dtype.kind not in "iu"
        or second.dtype.kind not in "iu"
        or labels.dtype != np.dtype(bool)
    ):
        raise ValueError(f"Некорректные индексы или метки: {scenario}")

    if (
        np.any(first < 0)
        or np.any(second < 0)
        or np.any(first >= record_count)
        or np.any(second >= record_count)
        or np.any(first == second)
    ):
        raise ValueError(f"Недопустимые индексы: {scenario}")

    canonical_pairs = np.sort(
        np.column_stack((first, second)),
        axis=1,
    )

    if len(np.unique(canonical_pairs, axis=0)) != len(first):
        raise ValueError(f"Повторные пары: {scenario}")

    if not (
        np.all(validation_conditions[first] == first_condition)
        and np.all(validation_conditions[second] == second_condition)
    ):
        raise ValueError(f"Не совпали условия: {scenario}")

    expected_labels = (
        validation_speakers[first] == validation_speakers[second]
    )

    if not np.array_equal(labels, expected_labels):
        raise ValueError(f"Не совпали метки дикторов: {scenario}")

    if not labels.any() or labels.all():
        raise ValueError(f"Нужны оба класса: {scenario}")

    scores = np.sum(
        validation_embeddings[first] * validation_embeddings[second],
        axis=1,
    ).astype(np.float64)

    validation_scores[scenario] = {
        "scores": scores,
        "labels": labels,
    }

    print(
        f"{scenario:<12} "
        f"SAME: {int(labels.sum()):>4} | "
        f"DIFFERENT: {int((~labels).sum()):>4}"
    )

print(f"\nМаксимальное отличие от кеша: {cache_difference:.8f}")
print("Validation-записей:", record_count)
print("Validation-дикторов:", len(set(validation_speakers.tolist())))
print("Данные готовы. Test не использовался.")

del sample_audio, current_embedding

Рассматриваем пороги между соседними уникальными score и два крайних решения: всё SAME и всё DIFFERENT. Для каждого порога считаем FAR/FRR отдельно по сценариям. Эта таблица позволит менять цену ошибок без повторного инференса.

In [ ]:
def build_threshold_table(score_data):
    scenarios = ("clean-clean", "clean-noisy", "noisy-noisy")

    if set(score_data) != set(scenarios):
        raise ValueError("Нужны ровно три validation-сценария")

    checked_data = {}

    for scenario in scenarios:
        scores = np.asarray(score_data[scenario]["scores"])
        labels = np.asarray(score_data[scenario]["labels"])

        if (
            scores.ndim != 1
            or scores.size == 0
            or scores.shape != labels.shape
            or scores.dtype.kind != "f"
            or labels.dtype != np.dtype(bool)
        ):
            raise ValueError(f"Некорректные массивы: {scenario}")

        if (
            not np.isfinite(scores).all()
            or np.any(np.abs(scores) > 1.00001)
        ):
            raise ValueError(f"Некорректные cosine-score: {scenario}")

        if not labels.any() or labels.all():
            raise ValueError(f"Нужны оба класса: {scenario}")

        checked_data[scenario] = (
            scores.astype(np.float64, copy=True),
            labels.copy(),
        )

    unique_scores = np.unique(np.concatenate([
        scores for scores, labels in checked_data.values()
    ]))

    thresholds = np.concatenate([
        [np.nextafter(unique_scores[0], -np.inf)],
        unique_scores[:-1] + np.diff(unique_scores) / 2,
        [np.nextafter(unique_scores[-1], np.inf)],
    ])

    far = np.empty((len(scenarios), len(thresholds)))
    frr = np.empty_like(far)

    for row, scenario in enumerate(scenarios):
        scores, labels = checked_data[scenario]

        positive = np.sort(scores[labels])
        negative = np.sort(scores[~labels])

        # SAME только при score > threshold.
        far[row] = (
            len(negative)
            - np.searchsorted(negative, thresholds, side="right")
        ) / len(negative)

        frr[row] = np.searchsorted(
            positive, thresholds, side="right"
        ) / len(positive)

        # Контроль быстрых расчётов прямым подсчётом.
        for column in np.linspace(
            0, len(thresholds) - 1, 9, dtype=int
        ):
            predictions = scores > thresholds[column]

            if not (
                np.isclose(
                    far[row, column],
                    predictions[~labels].mean(),
                )
                and np.isclose(
                    frr[row, column],
                    (~predictions[labels]).mean(),
                )
            ):
                raise RuntimeError(
                    f"Не совпал прямой подсчёт ошибок: {scenario}"
                )

    if not (
        np.all(far[:, 0] == 1)
        and np.all(frr[:, 0] == 0)
        and np.all(far[:, -1] == 0)
        and np.all(frr[:, -1] == 1)
    ):
        raise RuntimeError("Некорректные крайние решения")

    if (
        np.any(np.diff(thresholds) <= 0)
        or np.any(np.diff(far, axis=1) > 1e-12)
        or np.any(np.diff(frr, axis=1) < -1e-12)
    ):
        raise RuntimeError("Нарушена монотонность таблицы порогов")

    for array in (thresholds, far, frr):
        array.setflags(write=False)

    return {
        "scenarios": scenarios,
        "thresholds": thresholds,
        "far": far,
        "frr": frr,
    }


# При неудачном повторном запуске старая таблица недоступна.
threshold_table = None
threshold_table = build_threshold_table(validation_scores)

print("Порогов-кандидатов:", len(threshold_table["thresholds"]))
print("Сценариев:", len(threshold_table["scenarios"]))
print("Проверены крайние решения, монотонность и прямой подсчёт.")
print("Таблица готова. Test не использовался.")

Выбираем порог, минимизирующий ожидаемую стоимость: C_FA × P(DIFFERENT) × FAR + C_FR × P(SAME) × FRR. При точном равенстве стоимости выбираем больший порог. Все расчёты выполняются только по validation

In [ ]:
def select_operating_point(
    false_accept_cost=1.0,
    false_reject_cost=1.0,
    same_prior=0.5,
):
    parameters = {
        "false_accept_cost": false_accept_cost,
        "false_reject_cost": false_reject_cost,
        "same_prior": same_prior,
    }

    for name, value in parameters.items():
        if (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
            or not math.isfinite(value)
        ):
            raise ValueError(f"Некорректный параметр: {name}")

    if not (
        0 < false_accept_cost <= 100
        and 0 < false_reject_cost <= 100
    ):
        raise ValueError("Цены ошибок должны быть в диапазоне (0, 100]")

    if not 0 < same_prior < 1:
        raise ValueError("Вероятность SAME должна быть между 0 и 1")

    if threshold_table is None:
        raise RuntimeError("Сначала построй таблицу порогов")

    mean_far = threshold_table["far"].mean(axis=0)
    mean_frr = threshold_table["frr"].mean(axis=0)

    risks = (
        false_accept_cost * (1 - same_prior) * mean_far
        + false_reject_cost * same_prior * mean_frr
    )

    if not np.isfinite(risks).all():
        raise RuntimeError("Получены некорректные значения стоимости")

    # При точном равенстве выбираем больший порог.
    best = len(risks) - 1 - int(np.argmin(risks[::-1]))

    scenarios = threshold_table["scenarios"]

    return {
        "threshold": float(threshold_table["thresholds"][best]),
        "false_accept_cost": float(false_accept_cost),
        "false_reject_cost": float(false_reject_cost),
        "same_prior": float(same_prior),
        "different_prior": float(1 - same_prior),
        "scenario_weights": {
            scenario: 1 / len(scenarios)
            for scenario in scenarios
        },
        "validation_far": float(mean_far[best]),
        "validation_frr": float(mean_frr[best]),
        "validation_risk": float(risks[best]),
        "per_scenario": {
            scenario: {
                "far": float(threshold_table["far"][row, best]),
                "frr": float(threshold_table["frr"][row, best]),
            }
            for row, scenario in enumerate(scenarios)
        },
        "decision_rule": "similarity > threshold",
        "tie_break": "highest threshold among exact risk minima",
        "selection_split": "validation",
    }


def operating_point_from_slider(balance):
    if (
        isinstance(balance, bool)
        or not isinstance(balance, (int, float))
        or not math.isfinite(balance)
        or not -2 <= balance <= 2
    ):
        raise ValueError("Положение ползунка должно быть от -2 до 2")

    return select_operating_point(
        false_accept_cost=10 ** max(balance, 0),
        false_reject_cost=10 ** max(-balance, 0),
        same_prior=0.5,
    )


print("Баланс      C_FA      C_FR       Порог       FAR       FRR")

for balance in (-2, -1, 0, 1, 2):
    point = operating_point_from_slider(balance)

    print(
        f"{balance:>6}"
        f"{point['false_accept_cost']:>10.2f}"
        f"{point['false_reject_cost']:>10.2f}"
        f"{point['threshold']:>12.6f}"
        f"{point['validation_far']:>10.2%}"
        f"{point['validation_frr']:>10.2%}"
    )

slider_thresholds = np.array([
    operating_point_from_slider(float(balance))["threshold"]
    for balance in np.linspace(-2, 2, 81)
])

if np.any(np.diff(slider_thresholds) < 0):
    raise RuntimeError(
        "Порог уменьшился при увеличении цены ложного совпадения"
    )

default_operating_point = operating_point_from_slider(0)

print("\nРежим по умолчанию: одинаковая цена ошибок.")
print("Монотонность порога при изменении ползунка проверена.")
print("Исходный порог калибратора не изменён.")

Цена ошибок влияет на порог решения, но не меняет калиброванную оценку p_same для одной пары. Confidence относится к выданному ответу и может быть ниже 0.5, поскольку рабочий порог выбирается отдельно. Доступ к модели ограничиваем блокировкой.

In [ ]:
def compare_audio(first_path, second_path, balance=0.0):
    operating_point = operating_point_from_slider(balance)

    first_audio, first_duration = load_audio(first_path)
    second_audio, second_duration = load_audio(second_path)

    with inference_lock, torch.inference_mode():
        first_vector = extract_embedding(first_audio)
        second_vector = extract_embedding(second_audio)

        similarity = torch.dot(
            first_vector,
            second_vector,
        ).item()

    if not math.isfinite(similarity):
        raise RuntimeError("Получено некорректное сходство записей")

    logit = calibration_a * similarity + calibration_b

    if not math.isfinite(logit):
        raise RuntimeError("Получено некорректное значение калибратора")

    p_same = torch.sigmoid(
        torch.tensor(logit, dtype=torch.float64)
    ).item()

    selected_threshold = operating_point["threshold"]
    same_speaker = similarity > selected_threshold
    confidence = p_same if same_speaker else 1 - p_same

    notes = [
        "Confidence предварительный: калибровка на Vibravox "
        "при равном весе классов и сценариев.",
        "Наличие речи, число говорящих и подлинность голоса "
        "автоматически не проверяются.",
    ]

    if min(first_duration, second_duration) < 3:
        notes.append(
            "Есть запись короче 3 секунд: качество ниже, "
            "калибровка confidence для такой длины не подтверждена."
        )

    if confidence < 0.5:
        notes.append(
            "Confidence ниже 0.5: выбранный по цене ошибок порог "
            "не обязан соответствовать p_same = 0.5."
        )

    return {
        "label": "SAME" if same_speaker else "DIFFERENT",
        "same_speaker": bool(same_speaker),
        "confidence": round(confidence, 6),
        "p_same": round(p_same, 6),
        "similarity": round(similarity, 6),
        "threshold": selected_threshold,
        "false_accept_cost": operating_point["false_accept_cost"],
        "false_reject_cost": operating_point["false_reject_cost"],
        "assumed_same_prior": operating_point["same_prior"],
        "duration_seconds": [
            round(first_duration, 3),
            round(second_duration, 3),
        ],
        "model": "ECAPA Vibravox classification step 500",
        "model_sha256": artifact_manifest[
            "checkpoints/vibravox_fc_classification_1200/best.pt"
        ][1],
        "notes": notes,
    }


print("compare_audio готова.")
print("По умолчанию balance=0: одинаковая цена ошибок.")
print("Веса модели и сохранённые настройки не изменяются.")

Это функциональная проверка на двух парах, а не новая оценка качества. Проверяем формат ответа, соответствие решения порогу и симметрию сравнения. Ошибка распознавания отдельной пары возможна и сама по себе не означает ошибку реализации.

In [ ]:
def validation_audio_path(index):
    path = Path(str(validation_paths[index])).resolve(strict=True)

    if not path.is_relative_to(validation_audio_dir):
        raise ValueError("Запись находится вне каталога validation")

    return path


def check_comparison_result(result):
    required = {
        "label",
        "same_speaker",
        "confidence",
        "p_same",
        "similarity",
        "threshold",
    }

    if not required.issubset(result):
        raise RuntimeError("В ответе отсутствуют обязательные поля")

    if type(result["same_speaker"]) is not bool:
        raise RuntimeError("same_speaker должен иметь тип bool")

    expected_same = result["similarity"] > result["threshold"]
    expected_label = "SAME" if expected_same else "DIFFERENT"

    if (
        result["same_speaker"] != expected_same
        or result["label"] != expected_label
    ):
        raise RuntimeError("Ответ не соответствует выбранному порогу")

    for key in ("confidence", "p_same"):
        value = result[key]

        if not math.isfinite(value) or not 0 <= value <= 1:
            raise RuntimeError(f"Некорректное поле: {key}")

    expected_confidence = (
        result["p_same"]
        if result["same_speaker"]
        else 1 - result["p_same"]
    )

    if not math.isclose(
        result["confidence"],
        expected_confidence,
        rel_tol=0,
        abs_tol=2e-6,
    ):
        raise RuntimeError("Confidence не соответствует ответу")


clean_labels = validation_pair_indices["clean-clean_labels"]

print(
    f"{'Пара':<12} {'Ожидаем':<12} {'Получили':<12}"
    f"{'Similarity':>12} {'Confidence':>12}"
)

for expected_same in (True, False):
    pair_index = int(np.flatnonzero(clean_labels == expected_same)[0])

    first_index = int(
        validation_pair_indices["clean-clean_first"][pair_index]
    )
    second_index = int(
        validation_pair_indices["clean-clean_second"][pair_index]
    )

    first_path = validation_audio_path(first_index)
    second_path = validation_audio_path(second_index)

    forward = compare_audio(first_path, second_path, balance=0)
    reverse = compare_audio(second_path, first_path, balance=0)

    check_comparison_result(forward)
    check_comparison_result(reverse)

    if not math.isclose(
        forward["similarity"],
        reverse["similarity"],
        rel_tol=0,
        abs_tol=1e-5,
    ):
        raise RuntimeError("Сходство зависит от порядка файлов")

    if forward["label"] != reverse["label"]:
        raise RuntimeError("Бинарный ответ зависит от порядка файлов")

    expected_label = "SAME" if expected_same else "DIFFERENT"

    print(
        f"{'Один диктор' if expected_same else 'Разные':<12} "
        f"{expected_label:<12} "
        f"{forward['label']:<12}"
        f"{forward['similarity']:>12.6f}"
        f"{forward['confidence']:>12.6f}"
    )

for invalid_balance in (-3, 3, float("nan"), float("inf"), True):
    try:
        operating_point_from_slider(invalid_balance)
    except ValueError:
        pass
    else:
        raise RuntimeError(
            "Некорректное положение ползунка не было отклонено"
        )

print("\nФормат ответа и симметрия проверены.")
print("Некорректные значения ползунка отклонены.")
print("Test не использовался.")

Сохраняем выбранный порог вместе с ценами ошибок, долями классов, validation-метриками и хешами артефактов. Каждый режим получает отдельный файл. Сначала записываем временный файл, затем переименовываем его в JSON, чтобы незавершённая запись не выглядела готовым результато

In [ ]:

def operating_point_summary(balance):
    point = operating_point_from_slider(balance)

    return (
        f"**Цена ложного совпадения:** "
        f"{point['false_accept_cost']:.2f}  \n"
        f"**Цена ложного отказа:** "
        f"{point['false_reject_cost']:.2f}  \n"
        f"**Порог:** {point['threshold']:.6f}  \n"
        f"**Validation FAR:** {point['validation_far']:.2%} · "
        f"**Validation FRR:** {point['validation_frr']:.2%}  \n"
    )


def save_operating_point(balance):
    point = operating_point_from_slider(balance)

    payload = {
        "schema_version": 1,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "slider_balance": float(balance),
        "operating_point": point,
        "model": {
            "name": "ECAPA Vibravox classification",
            "checkpoint_step": 500,
            "checkpoint_sha256": artifact_manifest[
                "checkpoints/vibravox_fc_classification_1200/best.pt"
            ][1],
        },
        "calibration": {
            "a": calibration_a,
            "b": calibration_b,
            "assumed_same_prior": calibration_config[
                "assumed_same_prior"
            ],
        },
        "artifact_sha256": {
            name: expected_hash
            for name, (_, expected_hash) in artifact_manifest.items()
        },
        "limitations": [
            "Порог выбран на validation.",
            "Validation-метрики не гарантируют качество новых запросов.",
            "Калибровка короткой речи и реального треска не подтверждена.",
        ],
    }

    # Сериализуем до создания файлов.
    content = json.dumps(
        payload,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

    output_dir = (
        project_dir / "outputs/operating_points"
    ).resolve()

    if not output_dir.is_relative_to(project_dir):
        raise RuntimeError("Каталог сохранения находится вне проекта")

    output_dir.mkdir(parents=True, exist_ok=True)

    final_path = output_dir / f"operating_point_{uuid4().hex}.json"
    temporary_path = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_dir,
            prefix=".operating_point_",
            suffix=".tmp",
            delete=False,
        ) as file:
            temporary_path = Path(file.name)
            file.write(content)
            file.flush()
            os.fsync(file.fileno())

        os.replace(temporary_path, final_path)

    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)

    return str(final_path)


print(operating_point_summary(0))
print("\nФункция сохранения готова. Файл пока не создавался.")

In [ ]:

ui_logger = logging.getLogger("speaker_verification.ui")


def ui_update_balance(balance):
    return operating_point_summary(balance), None, None


def ui_clear_result():
    return None


def ui_compare(first_path, second_path, balance):
    try:
        return compare_audio(first_path, second_path, balance)

    except ValueError as error:
        gr.Warning(str(error))
        return None

    except (OSError, RuntimeError):
        ui_logger.exception("Ошибка обработки аудиопары")
        gr.Warning(
            "Не удалось выполнить сравнение. "
            "Подробности записаны в журнал ядра Jupyter."
        )
        return None


def ui_save_mode(balance):
    try:
        return save_operating_point(balance)

    except ValueError as error:
        gr.Warning(str(error))
        return None

    except (OSError, RuntimeError):
        ui_logger.exception("Ошибка сохранения рабочего режима")
        gr.Warning(
            "Не удалось сохранить режим. "
            "Подробности записаны в журнал ядра Jupyter."
        )
        return None


if "demo" in globals():
    demo.close()

with gr.Blocks(title="Верификация диктора") as demo:
    gr.Markdown(
        "# Один диктор или разные?\n"
        "Загрузи два WAV длительностью **1–5 секунд**. "
        "В каждой записи должен говорить один человек. "
        "Предпочтительны **3–5 секунд речи**.\n\n"
        "Модель: ECAPA после адаптации на Vibravox, шаг 500. "
        "Без шумоподавления и удаления пауз."
    )

    balance_slider = gr.Slider(
        minimum=-2,
        maximum=2,
        value=0,
        step=0.1,
        label="Баланс цены ошибок: 0 — одинаковые цены",
    )

    gr.Markdown(
        "**Влево:** дороже ошибочно разделить одного диктора.  \n"
        "**Вправо:** дороже ошибочно объединить разных дикторов.  \n"
        "На краях соответствующая ошибка дороже в 100 раз."
    )

    mode_summary = gr.Markdown(
        value=operating_point_summary(0)
    )

    with gr.Row():
        first_upload = gr.File(
            label="Первая запись",
            file_types=[".wav"],
            type="filepath",
        )
        second_upload = gr.File(
            label="Вторая запись",
            file_types=[".wav"],
            type="filepath",
        )

    compare_button = gr.Button(
        "Сравнить",
        variant="primary",
    )
    comparison_output = gr.JSON(label="Результат")

    gr.Markdown(
        "Confidence — предварительная оценка. "
        "При изменении цены ошибок меняется порог решения, "
        "а не p_same одной и той же пары. "
        "Confidence выданного ответа может быть ниже 0.5.\n\n"
        "Наличие речи, число говорящих и подлинность голоса "
        "не проверяются. Короткие записи и реальный треск "
        "требуют отдельной проверки.\n\n"
        "Показанные FAR/FRR измерены на validation "
        "и не гарантируются для новых запросов."
    )

    save_button = gr.Button("Сохранить выбранный режим")
    saved_mode = gr.File(
        label="Режим и метаданные — JSON",
        interactive=False,
    )

    balance_slider.change(
        fn=ui_update_balance,
        inputs=balance_slider,
        outputs=[
            mode_summary,
            comparison_output,
            saved_mode,
        ],
        concurrency_id="verification_ui",
        concurrency_limit=1,
        api_name=False,
    )

    for upload in (first_upload, second_upload):
        upload.change(
            fn=ui_clear_result,
            inputs=None,
            outputs=comparison_output,
            concurrency_id="verification_ui",
            concurrency_limit=1,
            api_name=False,
        )

    compare_button.click(
        fn=ui_compare,
        inputs=[
            first_upload,
            second_upload,
            balance_slider,
        ],
        outputs=comparison_output,
        concurrency_id="verification_ui",
        concurrency_limit=1,
        api_name=False,
    )

    save_button.click(
        fn=ui_save_mode,
        inputs=balance_slider,
        outputs=saved_mode,
        concurrency_id="verification_ui",
        concurrency_limit=1,
        api_name=False,
    )

demo.queue(
    max_size=8,
    default_concurrency_limit=1,
)

demo.launch(
    server_name="127.0.0.1",
    share=False,
    inbrowser=True,
    max_file_size="5mb",
    show_error=False,
    show_api=False,
)